# ModelForge Lite — Phase 3: Base Model + RAG

This notebook adds retrieval to the same base model from Phase 2. Instead of answering from memory alone, the model now gets relevant reference answers pulled from a knowledge base and is asked to ground its response in them. This is Variant 2 of 4.

In [ ]:
!pip install -q transformers accelerate datasets huggingface_hub pandas sentence-transformers faiss-cpu

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 1. Pull the knowledge base and eval split

`knowledge_base.csv` (200 rows from Phase 1) becomes our searchable reference material — think of it as a stand-in for a company's support docs / past resolved tickets. It's completely separate from `eval.csv`, so nothing the model retrieves has ever been seen by the eval questions.

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

HF_USERNAME = "YOUR_HF_USERNAME"
dataset_repo = f"{HF_USERNAME}/modelforge-lite-support-data"

kb_path = hf_hub_download(repo_id=dataset_repo, filename="knowledge_base.csv", repo_type="dataset")
eval_path = hf_hub_download(repo_id=dataset_repo, filename="eval.csv", repo_type="dataset")

kb_df = pd.read_csv(kb_path)
eval_df = pd.read_csv(eval_path)
print(f"Knowledge base: {len(kb_df)} rows | Eval set: {len(eval_df)} rows")
kb_df.head()

## 2. Embed the knowledge base and build a FAISS index

We use a small, fast sentence-embedding model to turn each knowledge-base question+answer into a vector, then index all vectors with FAISS for fast similarity search. This is the entire "retrieval" mechanism — no external vector database needed.

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

kb_texts = (kb_df["instruction"] + " " + kb_df["response"]).tolist()  # adjust column names if different
kb_embeddings = embedder.encode(kb_texts, show_progress_bar=True, convert_to_numpy=True)

dimension = kb_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(kb_embeddings)
print(f"FAISS index built with {index.ntotal} vectors of dimension {dimension}")

## 3. Write the retrieval function

For a given question, this embeds it the same way and finds the top-k most similar knowledge-base entries.

In [ ]:
def retrieve(question, k=3):
    q_embedding = embedder.encode([question], convert_to_numpy=True)
    distances, indices = index.search(q_embedding, k)
    return kb_df.iloc[indices[0]]

# sanity check
sample = retrieve("How can I get a refund for my order?")
sample[["instruction", "response"]]

## 4. Load the base model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
print("Model loaded.")

## 5. Write the RAG generation function

The key difference from Phase 2: the system prompt now includes retrieved reference answers, and the model is explicitly instructed to ground its answer in them rather than invent one.

In [ ]:
import time

def build_rag_prompt(question, retrieved):
    context_block = "\n\n".join(
        f"Reference {i+1}:\nQ: {row['instruction']}\nA: {row['response']}"
        for i, (_, row) in enumerate(retrieved.iterrows())
    )
    system_prompt = (
        "You are a helpful customer support assistant. "
        "Use the reference Q&A pairs below to answer the customer's question. "
        "If the references don't cover the question, say what you can and note the limitation — "
        "do not invent policy details that aren't in the references.\n\n"
        f"{context_block}"
    )
    return system_prompt

def generate_rag_answer(question, k=3, max_new_tokens=150):
    retrieved = retrieve(question, k=k)
    system_prompt = build_rag_prompt(question, retrieved)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
        )
    latency = time.time() - start

    generated = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True)
    return answer.strip(), latency, retrieved["instruction"].tolist()

# sanity check
test_answer, test_latency, sources = generate_rag_answer("How can I get a refund for my order?")
print(f"Answer: {test_answer}\n\nLatency: {test_latency:.2f}s\nRetrieved sources: {sources}")

## 6. Run RAG over all eval questions

In [ ]:
results = []
for i, row in eval_df.iterrows():
    question = row["instruction"]
    answer, latency, sources = generate_rag_answer(question)
    results.append({
        "question": question,
        "intent": row.get("intent", ""),
        "rag_answer": answer,
        "rag_latency_sec": round(latency, 3),
        "retrieved_sources": " | ".join(sources),
    })
    print(f"[{i+1}/{len(eval_df)}] done")

rag_df = pd.DataFrame(results)
rag_df.to_csv("rag_results.csv", index=False)
rag_df.head()

## 7. Push results to Hugging Face

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="rag_results.csv",
    path_in_repo="results/rag_results.csv",
    repo_id=dataset_repo,
    repo_type="dataset",
)
print("Uploaded RAG results.")

## Done — Phase 3 checklist

- [ ] Knowledge base embedded and indexed with FAISS
- [ ] Retrieval function tested and returning sensible matches
- [ ] Base model answered all eval questions using retrieved context
- [ ] Pushed `rag_results.csv` to Hugging Face

Next: Phase 4 — LoRA fine-tuning.